# PathVLM-LiteBench Quick Start

This notebook provides a lightweight overview of PathVLM-LiteBench.

PathVLM-LiteBench is a CPU-compatible, laptop-GPU accelerated, low-compute toolkit for patch-level computational pathology vision-language model evaluation. It supports image-text retrieval, zero-shot classification, prompt sensitivity analysis, retrieval metrics, embedding caching, and visualization reports.

This notebook avoids downloading CLIP models by default. For full demos, run the command-line scripts in the `examples/` folder.

## Available Command-Line Demos

Patch-text retrieval:

```bash
python examples/01_patch_text_retrieval_demo.py --model clip --device auto
```

Zero-shot classification:

```bash
python examples/02_zero_shot_classification_demo.py --model clip --device auto
```

Prompt sensitivity analysis:

```bash
python examples/03_prompt_sensitivity_demo.py --model clip --device auto
```

Retrieval metrics smoke test:

```bash
python examples/04_retrieval_metrics_demo.py
```

## Inspect Registered Models

The model registry allows users to refer to models by short keys such as `clip`.

Pathology-specific models such as PLIP and CONCH are currently registered as placeholders for future extensions.

## Prompt Template Library

PathVLM-LiteBench includes built-in prompt templates for common pathology concepts.

Example usage:

```python
from pathvlm_litebench.prompts import get_prompt_variants, build_class_prompts

get_prompt_variants("tumor")
build_class_prompts(["tumor", "normal", "necrosis"])
```

Command-line usage:

```bash
python examples/02_zero_shot_classification_demo.py --model clip --device auto --class_names tumor normal necrosis
python examples/03_prompt_sensitivity_demo.py --model clip --device auto --use_pathology_prompts --concepts tumor normal necrosis
```

If no real pathology patch folder is passed, the demos still use generated RGB images as smoke tests. For meaningful CPath experiments, provide `--image_dir path/to/your_patch_folder`.

In [ ]:
from pathvlm_litebench.models import list_available_models, resolve_model_name

models = list_available_models()
models

In [ ]:
resolve_model_name("clip")

## Retrieval Metrics Toy Example

The following example uses toy embeddings to verify Recall@K computation.

It does not load any vision-language model.

In [ ]:
import torch
import torch.nn.functional as F

from pathvlm_litebench.evaluation import (
    compute_text_to_image_recall_at_k,
    compute_image_to_text_recall_at_k,
    compute_mean_recall,
)

image_embeddings = torch.tensor([
    [1.0, 0.0],
    [0.9, 0.1],
    [0.0, 1.0],
    [0.1, 0.9],
])

text_embeddings = torch.tensor([
    [1.0, 0.0],
    [0.0, 1.0],
])

image_embeddings = F.normalize(image_embeddings, p=2, dim=-1)
text_embeddings = F.normalize(text_embeddings, p=2, dim=-1)

text_to_image_positive_pairs = {
    0: {0, 1},
    1: {2, 3},
}

image_to_text_positive_pairs = {
    0: {0},
    1: {0},
    2: {1},
    3: {1},
}

t2i_metrics = compute_text_to_image_recall_at_k(
    image_embeddings=image_embeddings,
    text_embeddings=text_embeddings,
    positive_pairs=text_to_image_positive_pairs,
    k_values=(1, 2),
)

i2t_metrics = compute_image_to_text_recall_at_k(
    image_embeddings=image_embeddings,
    text_embeddings=text_embeddings,
    positive_pairs=image_to_text_positive_pairs,
    k_values=(1, 2),
)

t2i_metrics, i2t_metrics, compute_mean_recall(t2i_metrics)

## Notes

- The built-in demo images are not pathology images. They are only smoke tests.
- The default model key `clip` resolves to `openai/clip-vit-base-patch32`.
- PLIP and CONCH are currently placeholders in the registry.
- Model-based demos support `--device auto`, `--device cpu`, and `--device cuda`.
- `auto` uses CUDA if available and otherwise falls back to CPU.
- The metrics demo does not use a model and therefore does not need a device argument.
- WSI-level processing is intentionally not included in the core workflow yet.